# 07 - Custom Models

This notebook walks through the OCI Vision custom-model lifecycle and the
client surfaces you use once a model is trained. The lifecycle still requires
real OCI credentials, but the local demo mode now exercises the `model_id`
hooks so the calling pattern is easy to test offline.


## Setup

In [1]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

client = VisionClient(demo=True)
demo_model_id = 'ocid1.aivisionmodel.oc1..examplecustommodel'
print(f"Client ready (demo={client.is_demo})")
print(f"Demo model id: {demo_model_id}")
print('Note: training and deployment still require live OCI credentials.')

Client ready (demo=True)
Demo model id: ocid1.aivisionmodel.oc1..examplecustommodel
Note: training and deployment still require live OCI credentials.


## The custom model lifecycle

OCI custom models still follow the same high-level flow:

1. create a project
2. prepare labeled training data
3. train a classification or detection model
4. wait for the model to become active
5. call inference with the model OCID

The new part in this repo is the direct `model_id=` hook on the client and CLI.


In [2]:
# Step 1: Create a project (conceptual -- requires OCI SDK)
#
# import oci
# config = oci.config.from_file()
# vision_client = oci.ai_vision.AIServiceVisionClient(config)
#
# project_details = oci.ai_vision.models.CreateProjectDetails(
#     display_name="my-custom-classifier",
#     compartment_id="ocid1.compartment.oc1..xxxxx",
#     description="Custom classifier for product categories"
# )
#
# project = vision_client.create_project(project_details).data
# print(f"Project ID: {project.id}")

print("Step 1: Create a project")
print("  -> vision_client.create_project(project_details)")
print("  -> Returns project OCID for subsequent steps")

Step 1: Create a project
  -> vision_client.create_project(project_details)
  -> Returns project OCID for subsequent steps


In [3]:
# Step 2: Prepare training data
#
# Training data layout in Object Storage:
# oci://my-bucket/training-data/
#   class_a/
#     img001.jpg
#     img002.jpg
#   class_b/
#     img003.jpg
#     img004.jpg
#   manifest.json  <-- optional metadata file

print("Step 2: Prepare training data")
print("  -> Upload labeled images to OCI Object Storage")
print("  -> Organize by class in subdirectories")
print("  -> Minimum 10 images per class (50+ recommended)")
print()
print("Supported formats: JPEG, PNG, BMP, TIFF")
print("Recommended: 224x224 to 1024x1024 pixels")

Step 2: Prepare training data
  -> Upload labeled images to OCI Object Storage
  -> Organize by class in subdirectories
  -> Minimum 10 images per class (50+ recommended)

Supported formats: JPEG, PNG, BMP, TIFF
Recommended: 224x224 to 1024x1024 pixels


In [4]:
# Step 3: Create a model (training job)
#
# model_details = oci.ai_vision.models.CreateModelDetails(
#     display_name="product-classifier-v1",
#     compartment_id="ocid1.compartment.oc1..xxxxx",
#     project_id=project.id,
#     model_type="IMAGE_CLASSIFICATION",  # or OBJECT_DETECTION
#     training_dataset=oci.ai_vision.models.ObjectStorageDataset(
#         dataset_type="OBJECT_STORAGE",
#         namespace_name="my-namespace",
#         bucket_name="my-bucket",
#         object_name="training-data/manifest.json"
#     ),
#     max_training_duration_in_hours=2.0,
#     is_quick_mode=True  # faster training, good for prototyping
# )
#
# model = vision_client.create_model(model_details).data
# print(f"Training started: {model.id}")

print("Step 3: Create and train the model")
print("  -> vision_client.create_model(model_details)")
print("  -> Model types: IMAGE_CLASSIFICATION or OBJECT_DETECTION")
print("  -> Quick mode: ~15-30 min; Full mode: 1-24 hours")
print("  -> Training is asynchronous -- poll for ACTIVE status")

Step 3: Create and train the model
  -> vision_client.create_model(model_details)
  -> Model types: IMAGE_CLASSIFICATION or OBJECT_DETECTION
  -> Quick mode: ~15-30 min; Full mode: 1-24 hours
  -> Training is asynchronous -- poll for ACTIVE status


In [5]:
# Step 4: Check training status and metrics
#
# model = vision_client.get_model(model_id).data
# print(f"Status: {model.lifecycle_state}")
# print(f"Precision: {model.metrics.precision}")
# print(f"Recall: {model.metrics.recall}")
# print(f"F1 Score: {model.metrics.f1_score}")

# Simulated training metrics
print("Step 4: Training metrics (example)")
print()
metrics = {
    "Status": "ACTIVE",
    "Precision": 0.943,
    "Recall": 0.921,
    "F1 Score": 0.932,
    "Training Time": "27 minutes",
    "Total Images": 500,
    "Classes": 5,
}
for key, val in metrics.items():
    print(f"  {key:20s}: {val}")

Step 4: Training metrics (example)

  Status              : ACTIVE
  Precision           : 0.943
  Recall              : 0.921
  F1 Score            : 0.932
  Training Time       : 27 minutes
  Total Images        : 500
  Classes             : 5


## Explore the results

### Calling a custom model with `VisionClient`

Once a custom model is active, you pass its OCID through `model_id=`.
In demo mode the OCID is ignored, but the API surface is the same and easy
to exercise locally.


In [6]:
custom_classification = client.classify('dog_closeup.jpg', model_id=demo_model_id)
custom_detection = client.detect_objects('dog_closeup.jpg', model_id=demo_model_id)

print('Classification labels:')
for label in custom_classification.labels[:5]:
    print(f'  {label.name}: {label.confidence_pct:.1f}%')

print() 
print('Detection objects:')
for obj in custom_detection.objects[:5]:
    print(f'  {obj.name}: {obj.confidence_pct:.1f}%')

Classification labels:
  Dog: 99.2%
  Vegetation: 98.8%
  Metal: 98.6%
  Fur: 98.2%
  Paw: 97.6%

Detection objects:
  Dog: 98.2%
  Dog: 97.9%
  Dog: 97.8%
  Bull: 95.5%
  Car: 93.3%


## Visualize

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

names = [label.name for label in custom_classification.labels[:8]]
confs = [label.confidence_pct for label in custom_classification.labels[:8]]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#E63946' if conf > 50 else '#A8DADC' for conf in confs]
ax.barh(names, confs, color=colors)
ax.set_xlabel('Confidence (%)')
ax.set_title('Custom model hook preview (demo response)')
ax.set_xlim(0, 105)
for i, conf in enumerate(confs):
    ax.text(conf + 0.5, i, f'{conf:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Under the hood

In [8]:
import json

payload = {
    'python_client': {
        'classification_call': f"client.classify('image.jpg', model_id='{demo_model_id}')",
        'detection_call': f"client.detect_objects('image.jpg', model_id='{demo_model_id}')",
    },
    'cli': {
        'classification_call': f"oci-vision classify image.jpg --model-id {demo_model_id}",
        'detection_call': f"oci-vision detect image.jpg --model-id {demo_model_id}",
    },
    'demo_preview': custom_classification.model_dump(),
}
print(json.dumps(payload, indent=2, default=str))

{
  "python_client": {
    "classification_call": "client.classify('image.jpg', model_id='ocid1.aivisionmodel.oc1..examplecustommodel')",
    "detection_call": "client.detect_objects('image.jpg', model_id='ocid1.aivisionmodel.oc1..examplecustommodel')"
  },
  "cli": {
    "classification_call": "oci-vision classify image.jpg --model-id ocid1.aivisionmodel.oc1..examplecustommodel",
    "detection_call": "oci-vision detect image.jpg --model-id ocid1.aivisionmodel.oc1..examplecustommodel"
  },
  "demo_preview": {
    "model_version": "1.5.97",
    "labels": [
      {
        "name": "Dog",
        "confidence": 0.9925129,
        "confidence_pct": 99.25
      },
      {
        "name": "Vegetation",
        "confidence": 0.9876839,
        "confidence_pct": 98.77
      },
      {
        "name": "Metal",
        "confidence": 0.9863116,
        "confidence_pct": 98.63
      },
      {
        "name": "Fur",
        "confidence": 0.9822804,
        "confidence_pct": 98.23
      },
      {


## Try it yourself

1. Train or deploy a real OCI Vision custom model.
2. Replace `demo_model_id` with your real OCID.
3. Switch to `VisionClient()` and call `classify(..., model_id=...)`.
4. Use the matching CLI path: `oci-vision classify image.jpg --model-id ...`.
5. Compare your custom-model output against a baseline pretrained run.
